# P12026 Part 2: Pandas for NLP Preprocessing

This notebook teaches how to use Pandas to prepare text data for NLP tasks.

By the end of this file, you will be able to:
- create and inspect a DataFrame
- remove duplicate rows
- clean and transform text columns
- apply useful Pandas functions before tokenization and modeling

## Section 1: Pandas Basics for Text Data

Pandas is a Python library for tabular data analysis. In NLP workflows, it is commonly used to manage datasets of documents, tweets, reviews, and labels.

### What is a DataFrame?

A DataFrame is a two-dimensional table with rows and columns.

In this example:
- each row is one text record
- columns store attributes such as the text, source, and label

In [8]:
import pandas as pd

data = {
    "tweet": [
        "Hello WORLD!",
        "Pandas is GREAT",
        "Python is FUN",
        "Hello WORLD!",
        "",
        "NLP with pandas is useful"
    ],
    "source": ["x", "x", "y", "x", "y","y"],
    "label": ["greeting", "statement", "statement", "greeting", "statement","statement"]
}

df = pd.DataFrame(data)
df

,tweet,source,label
0,Hello WORLD!,x,greeting
1,Pandas is GREAT,x,statement
2,Python is FUN,y,statement
3,Hello WORLD!,x,greeting
4,,y,statement
5,NLP with pandas is useful,y,statement


### Quick DataFrame Inspection

Before cleaning text, it is a good habit to inspect the table shape, columns, and first rows.

In [9]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (6, 3)
Columns: ['tweet', 'source', 'label']


,tweet,source,label
0,Hello WORLD!,x,greeting
1,Pandas is GREAT,x,statement
2,Python is FUN,y,statement
3,Hello WORLD!,x,greeting
4,,y,statement


### Dropping Duplicates

Duplicate text rows can bias analysis and model training.

Use `drop_duplicates()` to keep one copy of repeated rows.

In [10]:
print("Rows before deduplication:", len(df))
df_no_duplicates = df.drop_duplicates(subset=["tweet"]).copy()
print("Rows after deduplication:", len(df_no_duplicates))
df_no_duplicates

Rows before deduplication: 6
Rows after deduplication: 5


,tweet,source,label
0,Hello WORLD!,x,greeting
1,Pandas is GREAT,x,statement
2,Python is FUN,y,statement
4,,y,statement
5,NLP with pandas is useful,y,statement


### Other Useful Pandas Functions for NLP Prep

These functions are commonly used before tokenization:
- `str.lower()` to normalize letter case
- `str.replace(..., regex=True)` to remove punctuation
- `str.strip()` to remove extra spaces
- `replace(r"^\\s*$", pd.NA, regex=True)` to convert empty/whitespace-only text into missing values
- `dropna()` to remove rows with missing text values
- `value_counts()` to inspect class balance

In [12]:
clean_df = df_no_duplicates.copy()

clean_df["tweet"] = (
    clean_df["tweet"]
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

# Convert empty strings (or whitespace-only strings) to missing values.
clean_df["tweet"] = clean_df["tweet"].replace(r"^\s*$", pd.NA, regex=True)

clean_df = clean_df.dropna(subset=["tweet"])
clean_df

,tweet,source,label
0,hello world,x,greeting
1,pandas is great,x,statement
2,python is fun,y,statement
5,nlp with pandas is useful,y,statement


### Label Distribution Check

Checking label counts helps you understand if one class dominates the dataset.

In [5]:
clean_df["label"].value_counts()

label
statement    3
greeting     1
Name: count, dtype: int64

## Section 2: Text Preprocessing Pipeline

This section groups multiple preprocessing steps that are commonly applied before NLP modeling.

Current steps in this section:
- convert text to lowercase
- remove punctuation

More preprocessing steps will be added here in the next parts.

In [13]:
lambda_df = df_no_duplicates.copy()

# Convert each tweet to lowercase using a lambda expression
# split() -> lowercase each word -> join back into one string
lambda_df["tweet"] = lambda_df["tweet"].apply(
    lambda tweet: " ".join(word.lower() for word in tweet.split())
)

lambda_df[["tweet", "label"]]

,tweet,label
0,hello world!,greeting
1,pandas is great,statement
2,python is fun,statement
4,,statement
5,nlp with pandas is useful,statement


### Step 1: Lowercase Conversion (Lambda)

Explanation of the lambda expression:

- `df['tweet'].apply(...)`: applies a function to each row in the `tweet` column.
- `lambda tweet: ...`: defines a short function for each tweet.
- `tweet.split()`: splits the tweet string into words.
- `word.lower() for word in tweet.split()`: lowercases each word.
- `' '.join(...)`: joins the lowercase words back into a single sentence.

### Step 2: Remove Punctuation

Punctuation can add noise in NLP tasks (especially for bag-of-words or frequency-based models).

In this step, we remove punctuation while keeping letters, numbers, and spaces.

In [14]:
punctuation_df = lambda_df.copy()

# Remove punctuation using regex:
# [^\w\s] matches any character that is NOT a word character and NOT whitespace.
punctuation_df["tweet"] = punctuation_df["tweet"].str.replace(r"[^\w\s]", "", regex=True)

punctuation_df[["tweet", "label"]]

,tweet,label
0,hello world,greeting
1,pandas is great,statement
2,python is fun,statement
4,,statement
5,nlp with pandas is useful,statement


Why this regex works:

- `\w` = letters, digits, and underscore
- `\s` = whitespace (spaces, tabs, new lines)
- `[^\w\s]` = anything that is not a word character and not whitespace

So punctuation marks like `. , ! ? : ;` are removed.

In [15]:
# Alternative method using Python's built-in punctuation list
import string

alt_text = "I. like, this book!"
for c in string.punctuation:
    alt_text = alt_text.replace(c, "")

alt_text

'I like this book'

## Summary of This Part

In this notebook so far, you learned how to:
- build a DataFrame for text data
- inspect core table properties
- drop duplicates
- apply practical Pandas text-cleaning functions
- apply Section 2 preprocessing steps:
  - lowercase conversion
  - punctuation removal

This prepares the dataset for the next NLP steps such as tokenization, feature extraction, and modeling.